# 2.14 Cost Functions

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 2: Mathematics for Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-02-mathematics-for-machine-learning/02.14_cost-functions.ipynb)

Companion code for Section 2.14 of the book. Run the cells in order from top to bottom.

**Example 2.14.1**

In [ ]:
import numpy as np

y_true = np.array([3.0, 5.0, 7.0, 2.0, 8.0, 100.0])  # last is an outlier
y_pred = np.array([2.8, 5.2, 6.8, 2.5, 7.8,  10.0])

def mse(y, yh):   return np.mean((y-yh)**2)
def mae(y, yh):   return np.mean(np.abs(y-yh))
def rmse(y, yh):  return np.sqrt(mse(y,yh))
def huber(y, yh, delta=1.0):
    err = np.abs(y-yh)
    return np.where(err<=delta, 0.5*err**2, delta*(err - 0.5*delta)).mean()
def mape(y, yh):  return np.mean(np.abs((y-yh)/y))*100
def r2(y, yh):    return 1 - np.sum((y-yh)**2)/np.sum((y-y.mean())**2)

print('With outlier (y=100, ŷ=10):')
print(f'  MSE:   {mse(y_true,y_pred):.2f}  ← huge, dominated by outlier')
print(f'  MAE:   {mae(y_true,y_pred):.2f}  ← much smaller; outlier has less impact')
print(f'  RMSE:  {rmse(y_true,y_pred):.2f}')
print(f'  Huber: {huber(y_true,y_pred):.2f} ← between MSE and MAE')
print(f'  MAPE:  {mape(y_true,y_pred):.1f}%')
print(f'  R²:    {r2(y_true,y_pred):.4f}')

# Without outlier
y_t2 = y_true[:5]; y_p2 = y_pred[:5]
print(f'\nWithout outlier:')
print(f'  MSE:   {mse(y_t2,y_p2):.4f}   MAE: {mae(y_t2,y_p2):.4f}   R²: {r2(y_t2,y_p2):.4f}')

**Example 2.14.2**

In [ ]:
import numpy as np

# Binary Cross-Entropy Loss - the core loss for binary classification
def binary_ce(y_true, y_pred, eps=1e-15):
    y_pred = np.clip(y_pred, eps, 1-eps)   # numerical stability
    return -np.mean(y_true*np.log(y_pred) + (1-y_true)*np.log(1-y_pred))

y_true = np.array([1, 1, 0, 0, 1])

# Good predictions: high confidence, correct
y_good = np.array([0.95, 0.88, 0.05, 0.12, 0.92])
# Bad predictions: high confidence, WRONG
y_bad  = np.array([0.05, 0.12, 0.95, 0.88, 0.08])
# Uncertain predictions: always near 0.5
y_unc  = np.array([0.52, 0.48, 0.49, 0.51, 0.53])

print(f'Good predictions BCE:      {binary_ce(y_true, y_good):.4f}')
print(f'Uncertain predictions BCE:  {binary_ce(y_true, y_unc):.4f}')
print(f'Bad (confident+wrong) BCE:  {binary_ce(y_true, y_bad):.4f}')

# Why confident wrong predictions are catastrophically penalised
print(f'\n−log(0.95) = {-np.log(0.95):.4f}  (correct, confident)')
print(f'−log(0.50) = {-np.log(0.50):.4f}  (uncertain)')
print(f'−log(0.05) = {-np.log(0.05):.4f}  (wrong, confident) ← 30× worse!')

# Categorical cross-entropy with softmax output
def softmax(z):  return np.exp(z-z.max()) / np.exp(z-z.max()).sum()
def cat_ce(y_onehot, logits):
    probs = softmax(logits)
    return -np.sum(y_onehot * np.log(probs + 1e-15))

y_onehot = np.array([0, 1, 0, 0])   # True class = 1
logits_A = np.array([1.0, 3.5, 0.5, 1.0])  # High score for class 1 - good
logits_B = np.array([3.5, 0.5, 1.0, 2.0])  # High score for class 0 - bad
print(f'\nGood model CE:  {cat_ce(y_onehot, logits_A):.4f}')
print(f'Bad model CE:   {cat_ce(y_onehot, logits_B):.4f}')